# Codon Architecture of the Paralogous Connexin Genes *GJB2* and *GJB6* and a Complete Map of Their Synonymous Alternatives at the DFNB1 Locus

This notebook contains the complete computational workflow that reproduces every calculation, statistical test, table, and figure reported in the manuscript:

| Manuscript Section | Analysis Step | Notebook Cell | Required / Cached Input Files | Generated Outputs |
| :--- | :--- | :--- | :--- | :--- |
| **2.6 Software & Setup** | Environment & package verification | **Cell 1–2** | None | `run_metadata_manuscript.json` |
| **Figures (all)** | Publication figure settings (Springer): 174 mm width, height ≤ 234 mm, 8 pt lettering, lower-case panel letters, 600 dpi, 8-bit RGB TIFF + PNG + PDF (`save_figure`) | **Cell 2b** | None | — |
| **2.1 & 3.1 Reference Sequences, Codon Usage & tRNA Adaptation** | RefSeq CDS validation (`NM_004004.6`, `NM_006783.5`), GC/GC1/GC2/GC3, gene-specific RSCU, GENCODE v50 MANE Select `wCAI` & `CAI`, GtRNAdb hg38 `wtAI` & `tAI` | **Cell 3, 3b, 4** | `hg38-tRNAs.fa` (in `./gjb_saturation/`); auto-fetches & caches `NM_004004.6.gb`, `NM_006783.5.gb`, `gencode.v50.annotation.gtf.gz`, `gencode.v50.pc_transcripts.fa.gz` | `gene_metrics_MANE_v50.csv`, `genomewide_summary.json`, `tai_weights_by_codon.csv` |
| **3.1 Fig. 1** | Circular bar plots of codon counts & gene-specific RSCU in *GJB2* and *GJB6* | **Cell 5** | Uses `REF` and `RSCU_GENE` from Cell 3–4 | `Fig1.tif` / `.png` / `.pdf` |
| **2.2 & 3.2 Paralog Alignment, Domains & Permutation Test** | Biopython global pairwise alignment (`BLOSUM62`, gap `-10`/`-0.5`), UniProt `P29033` domain projection, **Table 1**, and 20,000-permutation topology test (`seed = 2026`) | **Cell 6** | Uses `REF`, `W_CAI_REF` | `Table1_Domain_Codon_Distribution.csv`, `topology_permutation_results.json` |
| **3.2 Fig. 2** | Positional alignment tracks (a) and domain proportions (b) | **Cell 7** | Uses alignment & domain annotations from Cell 6 | `Fig2.tif` / `.png` / `.pdf` |
| **2.4 & 3.4 Enumeration of Synonymous Alternatives & Local mRNA Folding** | Complete enumeration of 1,130 synonymous substitutions (952 SNVs, 178 MNVs), HGVS nomenclature, gene-specific `ΔRSCU`, `ΔwCAI`, `ΔwtAI`, 1,000 random wobble-penalty sets (`seed = 2026`), 81-nt ViennaRNA ensemble folding (`ΔGens`), the summary table reported in Section 3.4, and the full table of alternatives (source of **Online Resource 1**) | **Cell 8** | Uses `REF`, `RSCU_GENE`, `W_CAI_REF`, `TAI_WEIGHTS` | `Summary_Synonymous_Alternatives.csv`, `TableS1_all_alternatives.csv`, `tai_random_penalties.csv` |
| **3.4 Fig. 4** | Positional heatmaps of `ΔRSCU`, `ΔwCAI`, `ΔwtAI`, and `ΔGens` across *GJB2* and *GJB6* | **Cell 9** | Uses `swaps_df` from Cell 8 | `Fig4.tif` / `.png` / `.pdf` |
| **2.3 & 3.3 Primate Genome Alignments & Ancestral Codons** | Pairwise human–primate `LASTZ_NET` genome alignments across 23 primates + mouse outgroup (Ensembl release 116) and `EPO` + parsimony ancestral codon reconstruction | **Cell 10** | Uses `swaps_df` from Cell 8; caches Ensembl REST & `homo_sapiens_ancestor_13.fa` in `./primate_genome/cache/` | `species_summary_genome.csv`, `ancestral_codons.csv`, `alternatives_evolutionary_class.csv` |
| **3.3 Fig. 3** | Amino acid and codon identity across 23 non-human primate genomes and mouse outgroup | **Cell 11** | Uses `SUMM` from Cell 10 | `Fig3.tif` / `.png` / `.pdf` |
| **2.5 & 3.5 ClinVar Synonymous Variants** | Mapping 163 ClinVar synonymous SNVs (125 *GJB2*, 38 *GJB6*) onto the complete synonymous map and primate alignments; adds ClinVar annotations to the table of alternatives | **Cell 12** | `clinvar_candidates.csv` (in `./gjb_saturation/`) | `ClinVar_Synonymous_Atlas_Mapped.csv`, `TableS1_all_alternatives.csv`, `run_metadata_manuscript.json` |
| **Online Resources 1 & 2** | Electronic supplementary material: all 1,130 alternatives (Online Resource 1) and the 21 ClinVar synonymous variants with uncertain or conflicting classifications (Online Resource 2), each with article title, journal, authors and corresponding-author details | **Cell 13** | Uses `atlas` from Cell 12 | `ESM_1.xlsx`, `ESM_2.xlsx` |

**Figures:** all figures are drawn at their final printed size for *Molecular Genetics and Genomics* (174 mm double-column width, height ≤ 234 mm, 8 pt lettering) and saved at 600 dpi as 8-bit RGB TIFF (for submission), PNG (for insertion in the Word manuscript) and PDF, without cropping.

In [ ]:
# == Cell 1 == Dependencies & Software Environment (Methods 2.6) =================
!pip -q install "ViennaRNA==2.7.2" biopython scipy numpy pandas matplotlib requests openpyxl

import os, sys, re, math, json, time, gzip, glob, shutil, tarfile, hashlib, datetime, platform, urllib.request
from collections import Counter, defaultdict
from io import StringIO
import numpy as np
import pandas as pd
import scipy
import scipy.stats as stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.transforms as mtransforms
import requests

import RNA, Bio
from Bio import Entrez, SeqIO, Align
from Bio.Seq import Seq
from Bio.Align import substitution_matrices

print(f"Python {platform.python_version()} | NumPy {np.__version__} | pandas {pd.__version__} | "
      f"SciPy {scipy.__version__} | Biopython {Bio.__version__} | ViennaRNA {RNA.__version__} | "
      f"matplotlib {matplotlib.__version__}")

RUN_METADATA = {
    "manuscript_version": "MGG_submission_v1.1.0",
    "timestamp": datetime.datetime.now().isoformat(),
    "software": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "biopython": Bio.__version__,
        "ViennaRNA": RNA.__version__,
        "matplotlib": matplotlib.__version__,
    }
}


In [ ]:
# == Cell 2 == Configuration, Genomic Anchors & Genetic Code (Methods 2.1) ======
WORKDIR = "./gjb_saturation"
PRIMATE_DIR = "./primate_genome"
PRIMATE_CACHE = os.path.join(PRIMATE_DIR, "cache")
os.makedirs(WORKDIR, exist_ok=True)
os.makedirs(PRIMATE_CACHE, exist_ok=True)

ENTREZ_EMAIL = "your.name@example.com"   # replace with your own e-mail; used only if the RefSeq records are not found locally
TRANSCRIPTS = {"GJB2": "NM_004004.6", "GJB6": "NM_006783.5"}
EXPECTED_AA = {"GJB2": 226, "GJB6": 261}

# Local mRNA folding parameters (Methods 2.4): 81-nt window (40 nt flanking each side of middle codon nt)
PRIMARY_HALF = 40          # 2 * 40 + 1 = 81 nt window
FOLD_TEMP    = 37.0        # 37 °C
FOLD_DANGLES = 2           # dangles = 2
RANDOM_SEED  = 2026        # fixed seed for 20,000 topology permutations and 1,000 wobble perturbations
ATOL         = 1e-6

# GRCh38 genomic anchors on chromosome 13 (minus strand)
ASSEMBLY    = "GRCh38"
CHROM       = "13"
GENE_STRAND = {"GJB2": -1, "GJB6": -1}
CDS_G_START = {"GJB2": 20189581, "GJB6": 20223480}
COMP_TBL    = str.maketrans("ACGTacgtNn-.", "TGCAtgcaNn-.")

def cds_to_genomic(gene, cds_off0):
    """0-based CDS offset (0 = c.1) -> GRCh38 genomic coordinate on chr13."""
    return (CDS_G_START[gene] - cds_off0 if GENE_STRAND[gene] == -1
            else CDS_G_START[gene] + cds_off0)

# Standard genetic code table
_bases  = "TCAG"
_aas    = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"
_codons = [a + b + c for a in _bases for b in _bases for c in _bases]
CODON_TABLE = dict(zip(_codons, _aas))
SENSE_CODONS = [c for c, a in CODON_TABLE.items() if a != "*"]
SENSE_60 = [c for c in SENSE_CODONS if c != "ATG"]

AA2COD = defaultdict(list)
for _c, _a in CODON_TABLE.items():
    if _a != "*":
        AA2COD[_a].append(_c)
AA2COD = dict(AA2COD)
DEGEN_CODONS = [c for c in SENSE_CODONS if len(AA2COD[CODON_TABLE[c]]) > 1]

AA3 = dict(A="Ala", R="Arg", N="Asn", D="Asp", C="Cys", Q="Gln", E="Glu", G="Gly",
           H="His", I="Ile", L="Leu", K="Lys", M="Met", F="Phe", P="Pro", S="Ser",
           T="Thr", W="Trp", Y="Tyr", V="Val")

def translate_cds(cds):
    return "".join(CODON_TABLE[cds[i:i+3]] for i in range(0, len(cds) - 2, 3))

print(f"Configuration initialized. Sense codons: {len(SENSE_CODONS)}, degenerate sense codons: {len(DEGEN_CODONS)}.")
print(f"GRCh38 c.1 genomic anchors: {CDS_G_START}")


In [ ]:
# == Cell 2b == Publication Figure Settings (Molecular Genetics and Genomics) ======
# Springer artwork requirements: figure width 84 mm (single column) or 174 mm (double
# column), height at most 234 mm, lettering 8-12 pt at final size, figure parts
# labelled with lower-case letters, combination art at least 600 dpi.
# All figures are drawn at 174 mm width, so the font sizes below are the printed sizes.
from PIL import Image

CM = 1 / 2.54
FULL_W = 17.4 * CM        # 174 mm, double-column width (inches)
MAX_H  = 23.4 * CM        # 234 mm, maximum figure height (inches)
FIG_DPI = 600
FS = 8                    # base font size (pt) at final print size
FS_PANEL = 10             # panel letters (a, b)

matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "font.size": FS,
    "axes.titlesize": FS,
    "axes.labelsize": FS,
    "xtick.labelsize": FS,
    "ytick.labelsize": FS,
    "legend.fontsize": FS,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "pdf.fonttype": 42,          # editable text in PDF/EPS
    "ps.fonttype": 42,
    "savefig.facecolor": "white",
})

def panel_label(ax, letter, x=-0.08, y=1.02):
    """Lower-case bold panel letter (a, b, ...) as required by Springer."""
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=FS_PANEL, fontweight="bold",
            va="bottom", ha="left")

def save_figure(fig, stem):
    """Save Fig<n> as TIFF (600 dpi, 8-bit RGB, LZW), PNG (for Word) and PDF.
    The figure is saved at its exact drawn size (no tight cropping), so the
    printed width is exactly 174 mm."""
    w_in, h_in = fig.get_size_inches()
    assert abs(w_in - FULL_W) < 1e-6, "figure width must be 174 mm"
    assert h_in <= MAX_H + 1e-6, "figure height exceeds 234 mm"
    png = os.path.join(WORKDIR, stem + ".png")
    tif = os.path.join(WORKDIR, stem + ".tif")
    pdf = os.path.join(WORKDIR, stem + ".pdf")
    fig.savefig(png, dpi=FIG_DPI, facecolor="white")
    fig.savefig(pdf, facecolor="white")
    img = Image.open(png).convert("RGB")
    img.save(png, dpi=(FIG_DPI, FIG_DPI))
    img.save(tif, dpi=(FIG_DPI, FIG_DPI), compression="tiff_lzw")
    w_px, h_px = img.size
    print(f"Saved {stem}: {w_px} x {h_px} px, RGB, {FIG_DPI} dpi "
          f"= {w_px / FIG_DPI * 25.4:.0f} x {h_px / FIG_DPI * 25.4:.0f} mm")
    return png

print(f"Figure settings ready: width {FULL_W/CM*10:.0f} mm, max height {MAX_H/CM*10:.0f} mm, "
      f"{FIG_DPI} dpi, base font {FS} pt.")

In [ ]:
# == Cell 3 == Fetch & Validate Reference Transcripts (Methods 2.1, Results 3.1) ==
def validate_gb_record(rec, acc):
    if rec.id != acc:
        raise ValueError(f"Accession mismatch: {rec.id} != {acc}")
    cds_feat = next((f for f in rec.features if f.type == "CDS"), None)
    if not cds_feat:
        raise ValueError("No CDS feature found in GenBank record.")
    s, e = int(cds_feat.location.start), int(cds_feat.location.end)
    cds_seq = str(rec.seq[s:e]).upper()
    if len(cds_seq) % 3 != 0 or cds_seq[:3] != "ATG":
        raise ValueError("CDS fails reading frame or initiation codon (ATG) check.")
    return str(rec.seq).upper(), s, e - s

def fetch_refseq(acc, email):
    local_gb_path = os.path.join(WORKDIR, f"{acc}.gb")
    if os.path.exists(local_gb_path):
        rec = SeqIO.read(local_gb_path, "genbank")
        mrna, s, length = validate_gb_record(rec, acc)
        print(f"Loaded and validated cached GenBank record for {acc} ({local_gb_path}).")
        return rec.id, mrna, s, length

    print(f"Fetching {acc} from NCBI Entrez...")
    Entrez.email = email
    with Entrez.efetch(db="nuccore", id=acc, rettype="gb", retmode="text") as h:
        gb_data = h.read()
    rec = SeqIO.read(StringIO(gb_data), "genbank")
    mrna, s, length = validate_gb_record(rec, acc)
    with open(local_gb_path, "w") as f:
        f.write(gb_data)
    print(f"Fetched, validated, and cached {acc}.")
    return rec.id, mrna, s, length

REF = {}
for gene, acc in TRANSCRIPTS.items():
    acc_ret, mrna, cs, cl = fetch_refseq(acc, ENTREZ_EMAIL)
    cds = mrna[cs:cs + cl]
    prot = translate_cds(cds)
    assert len(cds) % 3 == 0 and cds[:3] == "ATG" and prot[-1] == "*" and prot[:-1].count("*") == 0
    assert len(prot) - 1 == EXPECTED_AA[gene]
    sense_codons = [cds[i:i+3] for i in range(0, len(cds) - 3, 3)]
    REF[gene] = dict(
        acc=acc_ret, mrna=mrna, cds_start=cs, cds_len=cl, cds=cds,
        prot=prot[:-1], n_sense_codons=len(sense_codons), codons=sense_codons,
        sha256=hashlib.sha256(mrna.encode()).hexdigest()
    )
    print(f"{gene} ({acc_ret}): CDS = {cl} nt ({len(sense_codons)} sense codons + 1 stop codon), "
          f"mRNA = {len(mrna)} nt (5'UTR={cs} nt, 3'UTR={len(mrna)-cs-cl} nt)")


In [ ]:
# == Cell 3b == Download GENCODE Release 50 Reference Files (Methods 2.1) =========
# GENCODE Human Release 50 (GRCh38): annotation and protein-coding transcript sequences,
# used in Cell 4 to derive genome-wide codon adaptation weights from MANE Select transcripts.
# Files are downloaded once and reused on later runs.
GENCODE_URL = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_50/"
for fname in ("gencode.v50.annotation.gtf.gz", "gencode.v50.pc_transcripts.fa.gz"):
    dest = os.path.join(WORKDIR, fname)
    if os.path.exists(dest):
        print(f"{fname} already present, skipping download.")
        continue
    print(f"Downloading {fname} (this may take a few minutes)...")
    urllib.request.urlretrieve(GENCODE_URL + fname, dest)
print("GENCODE Release 50 files ready.")

In [ ]:
# == Cell 4 == Codon Usage, GC1/GC2/GC3, Gene-Specific RSCU, wCAI & wtAI (3.1) ====
# 1. Position-specific GC contents and gene-specific RSCU across sense codons
RSCU_GENE = {}
CODON_COUNTS_GENE = {}

print("=== 3.1 Coding Sequence GC Content and Gene-Specific RSCU ===")
for gene in ("GJB2", "GJB6"):
    cods = REF[gene]["codons"]
    cds_sense = "".join(cods)
    gc_total = 100.0 * sum(b in "GC" for b in cds_sense) / len(cds_sense)
    gc1 = 100.0 * sum(c[0] in "GC" for c in cods) / len(cods)
    gc2 = 100.0 * sum(c[1] in "GC" for c in cods) / len(cods)
    gc3 = 100.0 * sum(c[2] in "GC" for c in cods) / len(cods)
    print(f"{gene}: {len(cods)} sense codons ({REF[gene]['cds_len']} nt w/ stop) | "
          f"Overall GC = {gc_total:.1f}% | GC1 = {gc1:.1f}%, GC2 = {gc2:.1f}%, GC3 = {gc3:.1f}%")

    cnt = Counter(cods)
    CODON_COUNTS_GENE[gene] = cnt
    rscu_g = {}
    for aa, fam in AA2COD.items():
        mean_cnt = sum(cnt[c] for c in fam) / len(fam)
        for c in fam:
            rscu_g[c] = (cnt[c] / mean_cnt) if mean_cnt > 0 else 0.0
    RSCU_GENE[gene] = rscu_g

print("\nKey Codon RSCU Comparison (GJB2 vs GJB6):")
for cod, aa_label in [("CTG", "Leu"), ("GTG", "Val"), ("ATC", "Ile"),
                      ("AAG", "Lys"), ("TTC", "Phe"), ("CAG", "Gln"),
                      ("AAC", "Asn"), ("AAT", "Asn")]:
    print(f"  {cod} ({aa_label}): GJB2 RSCU = {RSCU_GENE['GJB2'][cod]:.2f} (n={CODON_COUNTS_GENE['GJB2'][cod]}), "
          f"GJB6 RSCU = {RSCU_GENE['GJB6'][cod]:.2f} (n={CODON_COUNTS_GENE['GJB6'][cod]})")

absent_gjb2 = sorted(c for c in SENSE_CODONS if CODON_COUNTS_GENE["GJB2"][c] == 0)
absent_gjb6 = sorted(c for c in SENSE_CODONS if CODON_COUNTS_GENE["GJB6"][c] == 0)
absent_both = sorted(set(absent_gjb2) & set(absent_gjb6))
print(f"Absent sense codons -> GJB2 ({len(absent_gjb2)}): {absent_gjb2}")
print(f"Absent sense codons -> GJB6 ({len(absent_gjb6)}): {absent_gjb6}")
print(f"Absent from both genes ({len(absent_both)}): {absent_both}")

# 2. Genome-wide wCAI weights.
#    The frequencies below (per thousand codons) were computed once from the GENCODE release 50
#    MANE Select set (19,167 qualifying CDS; 11,131,402 codons) and are embedded here as a fixed
#    table so that the weights are identical on every run. When the GENCODE files are present,
#    section 4 below recomputes the family weights from them and reports the largest deviation.
HUMAN_CODON_FREQ = {
    'AAA':25.5954, 'AAC':18.6267, 'AAG':32.0341, 'AAT':17.2974,
    'ACA':15.1732, 'ACC':18.3253, 'ACG':5.9291,  'ACT':13.4486,
    'AGA':12.1163, 'AGC':19.8658, 'AGG':11.8009, 'AGT':12.6623,
    'ATA':7.6141,  'ATC':19.7326, 'ATG':21.2408, 'ATT':15.9339,
    'CAA':12.8427, 'CAC':15.0299, 'CAG':34.8172, 'CAT':11.1463,
    'CCA':17.3687, 'CCC':20.2143, 'CCG':7.3180,  'CCT':17.8911,
    'CGA':6.1029,  'CGC':10.3898, 'CGG':11.4593, 'CGT':4.4560,
    'CTA':7.1405,  'CTC':18.9315, 'CTG':39.1535, 'CTT':13.3217,
    'GAA':30.8858, 'GAC':25.1859, 'GAG':40.6800, 'GAT':22.4142,
    'GCA':16.0411, 'GCC':28.0131, 'GCG':7.6485,  'GCT':18.3140,
    'GGA':16.4643, 'GGC':22.2604, 'GGG':16.1625, 'GGT':10.5856,
    'GTA':7.1687,  'GTC':13.9488, 'GTG':27.3910, 'GTT':11.0293,
    'TAA':0.4883,  'TAC':14.5830, 'TAG':0.3861,  'TAT':12.0355,
    'TCA':12.7386, 'TCC':17.6014, 'TCG':4.5461,  'TCT':15.5114,
    'TGA':0.8475,  'TGC':12.1862, 'TGG':11.9483, 'TGT':10.6151,
    'TTA':7.9376,  'TTC':19.3375, 'TTG':12.9912, 'TTT':17.0732,
}
W_CAI_REF = {}
for aa, fam in AA2COD.items():
    mx = max(HUMAN_CODON_FREQ[c] for c in fam)
    for c in fam:
        W_CAI_REF[c] = HUMAN_CODON_FREQ[c] / mx

PREFERRED_CODONS = {c for c in DEGEN_CODONS if abs(W_CAI_REF[c] - 1.0) < 1e-9}
RARE_CODONS      = {c for c in DEGEN_CODONS if W_CAI_REF[c] < 0.5}

# 3. GtRNAdb hg38 tRNA Adaptation Weights (wtAI; dos Reis wobble model, 60-codon normalization)
TRNA_FILE = os.path.join(WORKDIR, "hg38-tRNAs.fa")
WOBBLE_S = {"I:U": 0.0, "I:C": 0.28, "I:A": 0.9999, "G:U": 0.41, "U:G": 0.68}
_WOBBLE_COMP = {"A": "T", "T": "A", "G": "C", "C": "G"}

# Fallback GtRNAdb hg38 elongator anticodon copy numbers (422 genes, 47 anticodons), used only
# if the FASTA is absent. The published results were produced from the FASTA.
DEFAULT_TGCN = {
    'AAC': 9, 'AAG': 9, 'AAT': 13, 'ACG': 7, 'AGA': 9, 'AGC': 22, 'AGG': 9, 'AGT': 8,
    'ATC': 3, 'ATT': 14, 'CAA': 7, 'CAG': 9, 'CAT': 10, 'CCA': 7, 'CCG': 4, 'CCT': 5,
    'CGA': 4, 'CGC': 5, 'CGG': 4, 'CGT': 6, 'CTA': 3, 'CTC': 13, 'CTG': 21, 'CTT': 15,
    'GAA': 10, 'GAC': 3, 'GAG': 5, 'GAT': 3, 'GCA': 29, 'GCC': 14, 'GCG': 5, 'GCT': 6,
    'GGA': 2, 'GGC': 1, 'GGG': 2, 'GGT': 5, 'GTA': 5, 'GTC': 18, 'GTG': 16, 'GTT': 20,
    'TAA': 5, 'TAC': 5, 'TAG': 6, 'TAT': 5, 'TCA': 4, 'TCC': 9, 'TCG': 6, 'TCT': 6
}

if os.path.exists(TRNA_FILE):
    tGCN = Counter()
    with open(TRNA_FILE, "r") as f:
        for line in f:
            if line.startswith(">"):
                m = re.search(r"tRNA-([A-Za-z]+)-([A-Za-z]{3})", line)
                if m and m.group(1).lower() not in ("undet", "pseudo", "sec", "sup", "imet"):
                    ac = m.group(2).upper().replace("U", "T")
                    if re.match(r"^[ACGT]{3}$", ac):
                        tGCN[ac] += 1
    print(f"\nParsed GtRNAdb FASTA ({TRNA_FILE}): {sum(tGCN.values())} elongator tRNA genes across {len(tGCN)} anticodons.")
else:
    tGCN = Counter(DEFAULT_TGCN)
    print(f"\nWARNING: {TRNA_FILE} not found. Falling back to the embedded GtRNAdb hg38 tGCN table "
          f"({sum(tGCN.values())} genes, {len(tGCN)} anticodons).")

def compute_tai_weights(tgcn_map, sIU=0.0, sGU=0.41, sIC=0.28, sIA=0.9999, sUG=0.68):
    def raw_w(codon):
        a2, a3, c3 = _WOBBLE_COMP[codon[1]], _WOBBLE_COMP[codon[0]], codon[2]
        g = lambda x: tgcn_map.get(x + a2 + a3, 0)
        if c3 == "T": return (1.0 - sIU) * g("A") + (1.0 - sGU) * g("G")
        if c3 == "C": return 1.0 * g("G") + (1.0 - sIC) * g("A")
        if c3 == "A": return 1.0 * g("T") + (1.0 - sIA) * g("A")
        return 1.0 * g("C") + (1.0 - sUG) * g("T")
    W_abs = {c: raw_w(c) for c in SENSE_CODONS}
    w_max = max(W_abs[c] for c in SENSE_60)
    w_norm = {c: W_abs[c] / w_max for c in SENSE_CODONS}
    nz = [w_norm[c] for c in SENSE_60 if w_norm[c] > 0]
    gm = math.exp(sum(map(math.log, nz)) / len(nz))
    return {c: (v if v > 0 else gm) for c, v in w_norm.items()}

TAI_WEIGHTS = compute_tai_weights(tGCN)
TAI_OPTIMAL_CODONS = {
    c for c in DEGEN_CODONS
    if TAI_WEIGHTS[c] >= max(TAI_WEIGHTS[x] for x in AA2COD[CODON_TABLE[c]]) - 1e-12
}

for gene in ("GJB2", "GJB6"):
    cods = REF[gene]["codons"]
    deg = [c for c in cods if c in DEGEN_CODONS]
    non_atg = [c for c in cods if c != "ATG"]
    n_met = cods.count("ATG")
    n_trp = cods.count("TGG")
    n_pref = sum(c in PREFERRED_CODONS for c in deg)
    n_rare = sum(c in RARE_CODONS for c in deg)
    n_tai_opt = sum(c in TAI_OPTIMAL_CODONS for c in deg)
    cai_val = math.exp(np.mean([math.log(W_CAI_REF[c]) for c in deg]))
    tai_val = math.exp(np.mean([math.log(TAI_WEIGHTS[c]) for c in non_atg]))
    REF[gene].update(dict(CAI=cai_val, tAI=tai_val, n_deg=len(deg)))
    print(f"{gene}: {len(deg)} multi-codon sense positions (excluding {n_met} Met + {n_trp} Trp = {n_met+n_trp}) | "
          f"Preferred (wCAI=1): {n_pref}/{len(deg)} ({100*n_pref/len(deg):.1f}%) | "
          f"Rare (wCAI<0.5): {n_rare}/{len(deg)} ({100*n_rare/len(deg):.1f}%) | "
          f"tAI-optimal: {n_tai_opt}/{len(deg)} ({100*n_tai_opt/len(deg):.1f}%) | "
          f"Gene CAI = {cai_val:.3f}, Gene tAI = {tai_val:.4f}")

# 4. Genome-wide MANE Select v50 context
GENCODE_URL = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_50/"
GTF_GZ = os.path.join(WORKDIR, "gencode.v50.annotation.gtf.gz")
PCT_GZ = os.path.join(WORKDIR, "gencode.v50.pc_transcripts.fa.gz")
MANE_CSV = os.path.join(WORKDIR, "gene_metrics_MANE_v50.csv")

if os.path.exists(MANE_CSV):
    gm_df = pd.read_csv(MANE_CSV).set_index("gene")
    print(f"\nLoaded cached MANE Select genome-wide table ({len(gm_df)} genes with >=50 multi-family codons).")
elif os.path.exists(GTF_GZ) and os.path.exists(PCT_GZ):
    mane = set()
    with gzip.open(GTF_GZ, "rt") as f:
        for line in f:
            if not line.startswith("#") and "\ttranscript\t" in line and 'tag "MANE_Select"' in line:
                mane.add(re.search(r'transcript_id "([^"]+)"', line).group(1))
    cds_mane = {}
    with gzip.open(PCT_GZ, "rt") as f:
        h, buf = None, []
        def _proc(hdr, seq):
            parts = hdr[1:].split("|"); tid, gname = parts[0], parts[5]
            if tid not in mane: return
            r = [x for x in parts if x.startswith("CDS:")]
            if not r: return
            a, b = map(int, r[0][4:].split("-")); s = seq[a-1:b].upper()
            if len(s) % 3 or not s.startswith("ATG"): return
            cs = [s[i:i+3] for i in range(0, len(s), 3)]
            if cs and CODON_TABLE.get(cs[-1]) == "*": cs = cs[:-1]
            if any(CODON_TABLE.get(c, "X") in ("*", "X") for c in cs): return
            cds_mane[gname] = cs
        for line in f:
            if line.startswith(">"):
                if h: _proc(h, "".join(buf))
                h, buf = line.strip(), []
            else:
                buf.append(line.strip())
        if h: _proc(h, "".join(buf))

    n_codons_total = sum(len(v) for v in cds_mane.values())
    print(f"\nParsed GENCODE v50 MANE Select: {len(cds_mane)} qualifying CDS; {n_codons_total} codons.")

    # Verify the embedded frequency table against these files. Family weights are ratios within
    # each synonymous family, so this comparison is independent of any normalisation.
    cnt_all = Counter()
    for cod in cds_mane.values():
        cnt_all.update(cod)
    w_recomputed = {}
    for aa, fam in AA2COD.items():
        mx = max(cnt_all[c] for c in fam)
        for c in fam:
            w_recomputed[c] = (cnt_all[c] / mx) if mx else 0.0
    max_dev = max(abs(w_recomputed[c] - W_CAI_REF[c]) for c in SENSE_CODONS)
    worst = max(SENSE_CODONS, key=lambda c: abs(w_recomputed[c] - W_CAI_REF[c]))
    print(f"  wCAI weight check against the embedded table: largest deviation = {max_dev:.4f} at {worst}.")
    if max_dev > 0.01:
        print("  WARNING: the embedded frequency table does not match these GENCODE files.")

    rows_m = []
    for g, cod in cds_mane.items():
        d = [c for c in cod if c in DEGEN_CODONS]
        if len(d) < 50: continue
        rows_m.append(dict(
            gene=g, n_codons=len(cod),
            CAI=float(np.exp(np.mean([np.log(W_CAI_REF[c]) for c in d]))),
            tAI=float(np.exp(np.mean([np.log(TAI_WEIGHTS[c]) for c in cod if c != "ATG"]))),
            GC3=float(np.mean([c[2] in "GC" for c in cod])),
            frac_preferred=float(np.mean([c in PREFERRED_CODONS for c in d]))
        ))
    gm_df = pd.DataFrame(rows_m).set_index("gene")
    for m in ["CAI", "tAI", "GC3", "frac_preferred"]:
        gm_df[m + "_pct"] = gm_df[m].rank(pct=True) * 100
    gm_df.to_csv(MANE_CSV)
else:
    raise FileNotFoundError(
        "Genome-wide context requires either gene_metrics_MANE_v50.csv, or both "
        "gencode.v50.annotation.gtf.gz and gencode.v50.pc_transcripts.fa.gz, in "
        f"{WORKDIR}. Download the two GENCODE release 50 files from {GENCODE_URL}"
    )

# Report and persist the genome-wide values quoted in Results 3.1
CONNEXIN_RE = re.compile(r"^GJ[ABCDE]\d+$")
cx = sorted(g for g in gm_df.index if CONNEXIN_RE.match(str(g)))

gw = {
    "n_genes": int(len(gm_df)),
    "connexins_in_set": cx,
    "genome_median": {m: round(float(gm_df[m].median()), 4)
                      for m in ["CAI", "tAI", "GC3", "frac_preferred"]},
}
print(f"\nMANE Select genome-wide context: {len(gm_df)} genes with >=50 degenerate codons; "
      f"median CAI = {gm_df['CAI'].median():.3f}; {len(cx)} connexin genes in the set")

for gene in ("GJB2", "GJB6"):
    if gene not in gm_df.index:
        raise KeyError(f"{gene} is missing from the MANE Select table; cannot report its percentiles.")
    row = gm_df.loc[gene]
    ent = {m: round(float(row[m]), 4) for m in ["CAI", "tAI", "GC3", "frac_preferred"]}
    for m in ["CAI", "tAI", "GC3", "frac_preferred"]:
        ent[m + "_percentile"] = round(float(row[m + "_pct"]), 1)
    for m in ["CAI", "GC3"]:
        r = int((gm_df.loc[cx, m] > row[m]).sum()) + 1
        ent["rank_among_connexins_" + m] = f"{r}/{len(cx)}"
    gw[gene] = ent
    print(f"  {gene}: CAI = {ent['CAI']:.3f} ({ent['CAI_percentile']:.1f}th percentile genome-wide, "
          f"{ent['rank_among_connexins_CAI']} connexins), tAI = {ent['tAI']:.4f} "
          f"({ent['tAI_percentile']:.1f}th percentile), GC3 = {100*ent['GC3']:.1f}%")

with open(os.path.join(WORKDIR, "genomewide_summary.json"), "w") as fh:
    json.dump(gw, fh, indent=1)

# Save tAI and wCAI weights per codon
pd.DataFrame({
    "codon": list(SENSE_CODONS),
    "tAI_weight": [TAI_WEIGHTS[c] for c in SENSE_CODONS],
    "wCAI_weight": [W_CAI_REF[c] for c in SENSE_CODONS]
}).to_csv(os.path.join(WORKDIR, "tai_weights_by_codon.csv"), index=False)
print("\nSaved genomewide_summary.json and tai_weights_by_codon.csv")

In [ ]:
# == Cell 5 == Fig. 1: Circular Bar Plots of Codon Usage in GJB2 & GJB6 (3.1) ======
# Drawn at 174 mm width. The two genes are stacked (GJB2 top, GJB6 bottom) so that all
# 61 spoke labels can be printed at 8 pt without overlapping. Codons of the same amino
# acid are grouped in alternating shaded sectors separated by gaps, with the amino acid
# letter on the outer edge of each sector. No titles or in-figure notes: the caption
# carries the description.
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

AA_ORDER = sorted(AA2COD.keys())
ORDERED_CODONS = [(aa, c) for aa in AA_ORDER for c in sorted(AA2COD[aa])]

# Angular layout: one slot per codon plus a gap between amino acid families
GAP = 0.35                                   # gap width in codon slots
slot_pos, pos, prev = [], 0.0, None
for aa, _ in ORDERED_CODONS:
    if prev is not None and aa != prev:
        pos += GAP
    slot_pos.append(pos + 0.5); pos += 1.0; prev = aa
TOTAL = pos + GAP                            # gap also between the last and first family
SLOT = 2 * np.pi / TOTAL
angles = np.array(slot_pos) * SLOT
FAMILY_SPAN = {}
for aa in AA_ORDER:
    idx = [i for i, (a, _) in enumerate(ORDERED_CODONS) if a == aa]
    FAMILY_SPAN[aa] = (angles[idx[0]] - 0.5 * SLOT, angles[idx[-1]] + 0.5 * SLOT)

fig = plt.figure(figsize=(FULL_W, MAX_H))
cmap = plt.cm.RdYlBu_r
norm = Normalize(vmin=0.0, vmax=4.0)

R_INNER = 6.0          # radius of the empty centre (count units)
R_LABEL = R_INNER + 15.5
R_SECT  = R_INNER + 23.6   # outer edge of the shaded sectors
R_AA    = R_INNER + 25.4
R_MAX   = R_INNER + 27.0

ax_pos = {"GJB2": [0.0, 0.500, 1.0, 0.500], "GJB6": [0.0, 0.000, 1.0, 0.500]}
for gene in ("GJB2", "GJB6"):
    ax = fig.add_axes(ax_pos[gene], projection="polar")
    ax.set_theta_zero_location("E")
    ax.set_theta_direction(1)

    # Alternating shaded sectors, one per amino acid family
    for k, aa in enumerate(AA_ORDER):
        a0, a1 = FAMILY_SPAN[aa]
        ax.bar((a0 + a1) / 2, R_SECT - R_INNER, width=a1 - a0, bottom=R_INNER,
               color="#ececec" if k % 2 == 0 else "#f8f8f8", edgecolor="none", zorder=0)
        ax.text((a0 + a1) / 2, R_AA, aa, fontsize=FS, fontweight="bold",
                ha="center", va="center", color="#111111")

    counts = [CODON_COUNTS_GENE[gene][c] for _, c in ORDERED_CODONS]
    rscus  = [RSCU_GENE[gene][c] for _, c in ORDERED_CODONS]
    colors = [cmap(norm(r)) for r in rscus]
    ax.bar(angles, counts, width=SLOT * 0.78, bottom=R_INNER, color=colors,
           edgecolor="#424242", lw=0.25, zorder=3)

    # Concentric reference rings at 4, 8 and 12 codons
    theta_ring = np.linspace(0, 2 * np.pi, 360)
    for r_val in (4, 8, 12):
        ax.plot(theta_ring, np.full_like(theta_ring, R_INNER + r_val), color="#9e9e9e",
                lw=0.35, ls=(0, (2, 2)), zorder=1)
        ax.text(np.deg2rad(-90), R_INNER + r_val, f"{r_val}", fontsize=FS, color="#424242",
                ha="center", va="center", zorder=5,
                bbox=dict(boxstyle="square,pad=0.08", fc="white", ec="none", alpha=0.85))
    ax.plot(theta_ring, np.full_like(theta_ring, R_INNER), color="#757575", lw=0.6, zorder=2)

    # Spoke labels: CODON (count)
    for ang, (aa, cod), cnt in zip(angles, ORDERED_CODONS, counts):
        deg = np.rad2deg(ang)
        right_side = (deg <= 90 or deg >= 270)
        ax.text(ang, R_LABEL, f"{cod} ({cnt})", rotation=deg if right_side else deg + 180,
                rotation_mode="anchor", ha="left" if right_side else "right", va="center",
                fontsize=FS, color="#212121", zorder=4)

    # Gene name (italic) and codon count in the centre
    ax.text(0, 0, rf"$\mathbf{{\mathit{{{gene}}}}}$" + f"\n{REF[gene]['n_sense_codons']} codons",
            ha="center", va="center", fontsize=FS, linespacing=1.2)
    ax.set_ylim(0, R_MAX)
    ax.axis("off")

cax = fig.add_axes([0.905, 0.40, 0.016, 0.20])
cb = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), cax=cax, orientation="vertical", ticks=[0, 1, 2, 3, 4])
cb.set_label("Gene-specific RSCU", fontsize=FS, labelpad=3)
cb.ax.tick_params(labelsize=FS, width=0.5, length=2)
cb.outline.set_linewidth(0.5)

fig1_out = save_figure(fig, "Fig1")
plt.show()

In [ ]:
# == Cell 6 == Paralog Alignment, Domains, Table 1 & Permutation Test (2.2, 3.2) ==
# 1. Global pairwise alignment of Cx26 (GJB2) and Cx30 (GJB6) using BLOSUM62 (gap open -10, extend -0.5)
aligner = Align.PairwiseAligner(mode="global", open_gap_score=-10.0, extend_gap_score=-0.5)
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aln = aligner.align(REF["GJB2"]["prot"], REF["GJB6"]["prot"])[0]

# Over codons 1..224, Cx26 and Cx30 align without gaps
ALN_LEN = 224
cx26_cods = REF["GJB2"]["codons"]
cx30_cods = REF["GJB6"]["codons"]
cx26_aa   = REF["GJB2"]["prot"]
cx30_aa   = REF["GJB6"]["prot"]

same_aa_pos  = [i for i in range(ALN_LEN) if cx26_aa[i] == cx30_aa[i]]
same_cod_pos = [i for i in same_aa_pos if cx26_cods[i] == cx30_cods[i]]
diff_cod_pos = [i for i in same_aa_pos if cx26_cods[i] != cx30_cods[i]]

# Exclude conserved Met/Trp when calculating proportion of shared codons among identical residues
same_aa_deg  = [i for i in same_aa_pos if cx26_cods[i] in DEGEN_CODONS]
same_cod_deg = [i for i in same_aa_deg if cx26_cods[i] == cx30_cods[i]]
diff_cod_deg = [i for i in same_aa_deg if cx26_cods[i] != cx30_cods[i]]

pref_in_shared = sum(cx26_cods[i] in PREFERRED_CODONS for i in same_cod_deg)
pref_in_div_g2 = sum(cx26_cods[i] in PREFERRED_CODONS for i in diff_cod_deg)
pref_in_div_g6 = sum(cx30_cods[i] in PREFERRED_CODONS for i in diff_cod_deg)

print("=== 3.2 Paralog Alignment Summary (Codons 1-224) ===")
print(f"Identical amino acids: {len(same_aa_pos)}/{ALN_LEN} ({100*len(same_aa_pos)/ALN_LEN:.1f}%) | "
      f"Same codon: {len(same_cod_pos)} | Different synonymous codon: {len(diff_cod_pos)}")
print(f"Excluding {len(same_aa_pos)-len(same_aa_deg)} conserved Met/Trp: "
      f"shared codon = {len(same_cod_deg)}/{len(same_aa_deg)} ({100*len(same_cod_deg)/len(same_aa_deg):.1f}%), "
      f"different codon = {len(diff_cod_deg)}/{len(same_aa_deg)} ({100*len(diff_cod_deg)/len(same_aa_deg):.1f}%)")
print(f"Preferred human codons (wCAI=1): {pref_in_shared}/{len(same_cod_deg)} ({100*pref_in_shared/len(same_cod_deg):.1f}%) of shared codons vs "
      f"{pref_in_div_g2}/{len(diff_cod_deg)} ({100*pref_in_div_g2/len(diff_cod_deg):.1f}%) in GJB2 and "
      f"{pref_in_div_g6}/{len(diff_cod_deg)} ({100*pref_in_div_g6/len(diff_cod_deg):.1f}%) in GJB6 at divergent sites.")

# 2. Topological domain boundaries from UniProt P29033 (Cx26) projected onto Cx30
DOMAINS = [
    ("NT",  1,  20,  20,  20, "Cytoplasmic"),
    ("TM1", 21, 40,  40,  40, "Transmembrane"),
    ("E1",  41, 73,  73,  73, "Extracellular"),
    ("TM2", 74, 94,  94,  94, "Transmembrane"),
    ("CL",  95, 135, 135, 135, "Cytoplasmic"),
    ("TM3", 136, 156, 156, 156, "Transmembrane"),
    ("E2",  157, 189, 189, 189, "Extracellular"),
    ("TM4", 190, 210, 210, 210, "Transmembrane"),
    ("CT",  211, 226, 261, 224, "Cytoplasmic"),
]

# Table 1: Preferred and rarely used reference codons across the topological domains of Cx26 and Cx30
t1_rows = []
for dname, s, e_g2, e_g6, e_aln, tclass in DOMAINS:
    g2_slice = cx26_cods[s-1:e_g2]
    g6_slice = cx30_cods[s-1:e_g6]
    g2_deg = [c for c in g2_slice if c in DEGEN_CODONS]
    g6_deg = [c for c in g6_slice if c in DEGEN_CODONS]
    g2_pref = sum(c in PREFERRED_CODONS for c in g2_deg)
    g6_pref = sum(c in PREFERRED_CODONS for c in g6_deg)
    g2_rare_list = [f"{AA3[CODON_TABLE[cx26_cods[i-1]]]}{i} {cx26_cods[i-1]}"
                    for i in range(s, e_g2 + 1) if cx26_cods[i-1] in RARE_CODONS]
    g6_rare_list = [f"{AA3[CODON_TABLE[cx30_cods[i-1]]]}{i} {cx30_cods[i-1]}"
                    for i in range(s, e_g6 + 1) if cx30_cods[i-1] in RARE_CODONS]
    label = f"{dname} ({s}–{e_g2})" if e_g2 == e_g6 else f"{dname} ({s}–{e_g2} / {e_g6})"
    t1_rows.append({
        "Domain": label,
        "GJB2 n": len(g2_deg),
        "GJB2 Preferred n (%)": f"{g2_pref} ({100*g2_pref/len(g2_deg):.1f})",
        "GJB2 Rarely used codons": ", ".join(g2_rare_list) if g2_rare_list else "–",
        "GJB6 n": len(g6_deg),
        "GJB6 Preferred n (%)": f"{g6_pref} ({100*g6_pref/len(g6_deg):.1f})",
        "GJB6 Rarely used codons": ", ".join(g6_rare_list) if g6_rare_list else "–",
    })

t1_rows.append({
    "Domain": "Total",
    "GJB2 n": 212,
    "GJB2 Preferred n (%)": "122 (57.5)",
    "GJB2 Rarely used codons": "18 codons",
    "GJB6 n": 247,
    "GJB6 Preferred n (%)": "125 (50.6)",
    "GJB6 Rarely used codons": "28 codons",
})
table1_df = pd.DataFrame(t1_rows)
table1_df.to_csv(os.path.join(WORKDIR, "Table1_Domain_Codon_Distribution.csv"), index=False)
print("\n=== Table 1: Preferred and Rarely Used Reference Codons Across Topological Domains ===")
print(table1_df.to_string(index=False))

# Shared rarely used codons at identical aligned positions
shared_rare = [f"{AA3[CODON_TABLE[cx26_cods[i]]]}{i+1} ({cx26_cods[i]})"
               for i in same_cod_deg if cx26_cods[i] in RARE_CODONS]
print(f"\nShared rarely used codons ({len(shared_rare)}/{len(same_cod_deg)} = "
      f"{100*len(shared_rare)/len(same_cod_deg):.1f}%): {', '.join(shared_rare)}")

# 3. Topology Permutation Test (20,000 permutations within each amino acid family, random seed 2026)
def topo_class_of(pos1):
    if (21 <= pos1 <= 40) or (74 <= pos1 <= 94) or (136 <= pos1 <= 156) or (190 <= pos1 <= 210):
        return "TM"
    if (41 <= pos1 <= 73) or (157 <= pos1 <= 189):
        return "E"
    return "C"

N_PERM = 20000
print(f"\n=== Topology Permutation Test ({N_PERM:,} permutations, seed = {RANDOM_SEED}) ===")
perm_summary = {}
for gene in ("GJB2", "GJB6"):
    cods = REF[gene]["codons"]
    n_c = len(cods)
    t_cls = [topo_class_of(i) for i in range(1, n_c + 1)]
    aa_idx = defaultdict(list)
    for i, c in enumerate(cods):
        if c in DEGEN_CODONS:
            aa_idx[CODON_TABLE[c]].append(i)

    exp_rare = {"TM": 0.0, "E": 0.0, "C": 0.0}
    exp_pref = {"TM": 0.0, "E": 0.0, "C": 0.0}
    exp_56_80 = 0.0
    exp_tm_leu = 0.0
    for aa, idxs in aa_idx.items():
        fam_cods = [cods[i] for i in idxs]
        pr = sum(c in RARE_CODONS for c in fam_cods) / len(fam_cods)
        pp = sum(c in PREFERRED_CODONS for c in fam_cods) / len(fam_cods)
        for i in idxs:
            exp_rare[t_cls[i]] += pr
            exp_pref[t_cls[i]] += pp
            if 56 <= (i + 1) <= 80:
                exp_56_80 += pr
            if t_cls[i] == "TM" and aa == "L":
                exp_tm_leu += pr

    obs_rare = {k: sum((cods[i] in RARE_CODONS) for i in range(n_c) if t_cls[i] == k and cods[i] in DEGEN_CODONS) for k in ("TM", "E", "C")}
    obs_pref = {k: sum((cods[i] in PREFERRED_CODONS) for i in range(n_c) if t_cls[i] == k and cods[i] in DEGEN_CODONS) for k in ("TM", "E", "C")}
    obs_56_80 = sum((cods[i] in RARE_CODONS) for i in range(55, 80) if cods[i] in DEGEN_CODONS)

    chi2 = lambda o, e: sum((o[k] - e[k])**2 / e[k] for k in ("TM", "E", "C"))
    obs_c2_rare = chi2(obs_rare, exp_rare)
    obs_c2_pref = chi2(obs_pref, exp_pref)

    rng = np.random.default_rng(RANDOM_SEED)
    c2_rare_perm = np.zeros(N_PERM)
    c2_pref_perm = np.zeros(N_PERM)
    pref_E_perm  = np.zeros(N_PERM)
    r56_80_perm  = np.zeros(N_PERM)

    for p in range(N_PERM):
        pr_d = {"TM": 0, "E": 0, "C": 0}
        pp_d = {"TM": 0, "E": 0, "C": 0}
        r56 = 0
        for aa, idxs in aa_idx.items():
            shuffled = rng.permutation([cods[i] for i in idxs])
            for i, c in zip(idxs, shuffled):
                cl = t_cls[i]
                if c in RARE_CODONS:
                    pr_d[cl] += 1
                    if 56 <= (i + 1) <= 80:
                        r56 += 1
                if c in PREFERRED_CODONS:
                    pp_d[cl] += 1
        c2_rare_perm[p] = chi2(pr_d, exp_rare)
        c2_pref_perm[p] = chi2(pp_d, exp_pref)
        pref_E_perm[p]  = pp_d["E"]
        r56_80_perm[p]  = r56

    p_rare = float(np.mean(c2_rare_perm >= obs_c2_rare - 1e-12))
    p_pref = float(np.mean(c2_pref_perm >= obs_c2_pref - 1e-12))
    p_pref_E = float(np.mean(np.abs(pref_E_perm - exp_pref["E"]) >= abs(obs_pref["E"] - exp_pref["E"]) - 1e-12))
    p_56_80 = float(np.mean(np.abs(r56_80_perm - exp_56_80) >= abs(obs_56_80 - exp_56_80) - 1e-12))

    perm_summary[gene] = dict(
        obs_rare=obs_rare, exp_rare={k: round(v, 2) for k, v in exp_rare.items()},
        obs_pref=obs_pref, exp_pref={k: round(v, 2) for k, v in exp_pref.items()},
        p_rare_overall=round(p_rare, 3), p_pref_overall=round(p_pref, 3),
        p_pref_bonferroni=round(min(1.0, p_pref * 4), 3),
        p_pref_E_two_sided=round(p_pref_E, 3),
        obs_rare_56_80=obs_56_80, exp_rare_56_80=round(exp_56_80, 2), p_rare_56_80=round(p_56_80, 2)
    )

    exp_rare_rnd = {k: round(v, 1) for k, v in exp_rare.items()}
    exp_pref_rnd = {k: round(v, 1) for k, v in exp_pref.items()}

    print(f"{gene}: Rare codons obs={obs_rare} vs exp={exp_rare_rnd} (TM Leu exp={exp_tm_leu:.1f}) -> P = {p_rare:.2f}")
    print(f"{gene}: Preferred codons obs={obs_pref} vs exp={exp_pref_rnd} -> "
          f"P = {p_pref:.3f} (Bonferroni-adjusted P = {min(1.0, p_pref*4):.3f}; E-loop 2-sided P = {p_pref_E:.3f})")
    print(f"{gene}: Residues 56-80 (35-40 codons downstream of TM1) rare obs={obs_56_80} vs exp={exp_56_80:.1f} -> P = {p_56_80:.2f}")

json.dump(perm_summary, open(os.path.join(WORKDIR, "topology_permutation_results.json"), "w"), indent=2)


In [ ]:
# == Cell 7 == Fig. 2: Paralog Alignment & Domain Codon Sharing (3.2) ===========
# Panels a and b. Drawn at 174 mm width.
BLUE  = "#1f77b4"   # Same amino acid, same codon
PINK  = "#e377c2"   # Same amino acid, different codon
WHITE = "#ffffff"   # Different amino acid

fig = plt.figure(figsize=(FULL_W, FULL_W * 0.68))
gs = fig.add_gridspec(2, 1, height_ratios=[1.0, 1.15], hspace=0.55,
                      left=0.14, right=0.985, top=0.95, bottom=0.10)

# Panel A: positional tracks along the alignment (1..261)
axA = fig.add_subplot(gs[0])
dom_colors = {"Cytoplasmic": "#dcdcdc", "Transmembrane": "#636363", "Extracellular": "#ffffff"}
for dname, s, e_g2, e_g6, e_aln, tclass in DOMAINS:
    end_x = 261 if dname == "CT" else e_aln
    axA.add_patch(mpatches.Rectangle((s - 0.5, 2.0), end_x - s + 1, 0.55,
                                     facecolor=dom_colors[tclass], edgecolor="#888888", lw=0.4))
    axA.text((s + end_x) / 2, 2.275, dname, ha="center", va="center", fontsize=FS,
             color="white" if tclass == "Transmembrane" else "#111111")

for i in range(1, ALN_LEN + 1):
    c2, c6 = cx26_cods[i-1], cx30_cods[i-1]
    a2, a6 = cx26_aa[i-1], cx30_aa[i-1]
    col_aa  = BLUE if c2 == c6 else (PINK if a2 == a6 else WHITE)
    col_cod = BLUE if c2 == c6 else WHITE
    axA.add_patch(mpatches.Rectangle((i - 0.5, 1.0), 1.0, 0.65, facecolor=col_aa, edgecolor="none"))
    axA.add_patch(mpatches.Rectangle((i - 0.5, 0.0), 1.0, 0.65, facecolor=col_cod, edgecolor="none"))

for y0 in (1.0, 0.0):
    axA.add_patch(mpatches.Rectangle((0.5, y0), ALN_LEN, 0.65, fill=False, edgecolor="#444444", lw=0.4))
    axA.add_patch(mpatches.Rectangle((224.5, y0), 261 - 224, 0.65, facecolor="#f5f5f5",
                                     edgecolor="#888888", hatch="////", lw=0.4))
axA.set_xlim(0.5, 261.5)
axA.set_ylim(-0.15, 2.7)
axA.set_yticks([0.325, 1.325, 2.275])
axA.set_yticklabels(["Same codon", "Same amino acid", "Domain"])
axA.tick_params(axis="y", length=0)
axA.set_xticks([1, 50, 100, 150, 200, 224, 261])
axA.set_xlabel("Codon position (Cx26/Cx30 alignment)")
for sp in ("top", "right", "left"):
    axA.spines[sp].set_visible(False)
panel_label(axA, "a", x=-0.155, y=1.06)

# Panel B: domain-level proportions (codons 1-224)
axB = fig.add_subplot(gs[1])
d_labels, p_same_cod, p_diff_cod, p_diff_aa, n_dom = [], [], [], [], []
for dname, s, e_g2, e_g6, e_aln, tclass in DOMAINS:
    n_d = e_aln - s + 1
    sc = sum(cx26_cods[i-1] == cx30_cods[i-1] for i in range(s, e_aln + 1))
    dc = sum((cx26_aa[i-1] == cx30_aa[i-1]) and (cx26_cods[i-1] != cx30_cods[i-1]) for i in range(s, e_aln + 1))
    da = n_d - sc - dc
    d_labels.append(dname); n_dom.append(n_d)
    p_same_cod.append(100 * sc / n_d); p_diff_cod.append(100 * dc / n_d); p_diff_aa.append(100 * da / n_d)

x_dom = np.arange(len(d_labels))
bar_kw = dict(width=0.62, edgecolor="#333333", lw=0.4)
axB.bar(x_dom, p_same_cod, color=BLUE, label="Same amino acid, same codon", **bar_kw)
axB.bar(x_dom, p_diff_cod, bottom=p_same_cod, color=PINK, label="Same amino acid, different codon", **bar_kw)
axB.bar(x_dom, p_diff_aa, bottom=np.array(p_same_cod) + np.array(p_diff_cod), color=WHITE,
        label="Different amino acid", **bar_kw)
for xi, nd in zip(x_dom, n_dom):
    axB.text(xi, 102, f"n={nd}", ha="center", va="bottom", fontsize=FS)
axB.set_xticks(x_dom)
axB.set_xticklabels(d_labels)
axB.set_ylim(0, 114)
axB.set_yticks([0, 25, 50, 75, 100])
axB.set_ylabel("Aligned positions (%)")
axB.legend(loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=3, frameon=False,
           handlelength=1.2, columnspacing=1.5)
for sp in ("top", "right"):
    axB.spines[sp].set_visible(False)
panel_label(axB, "b", x=-0.155, y=1.02)

fig2_out = save_figure(fig, "Fig2")
plt.show()

In [ ]:
# == Cell 8 == Enumeration of 1,130 Alternatives, 81-nt Folding, Wobble Sensitivity & Summary Table (2.4, 3.4) ==
def fold_ensemble_81nt(seq):
    md_ = RNA.md()
    md_.temperature = FOLD_TEMP
    md_.dangles = FOLD_DANGLES
    fc = RNA.fold_compound(seq.replace("T", "U"), md_)
    _, mfe_val = fc.mfe()
    fc.exp_params_rescale(mfe_val)
    _, gens_val = fc.pf()
    return mfe_val, gens_val

def extract_81nt_window(mrna_seq, middle_nt_idx0, half=PRIMARY_HALF):
    lo = max(0, middle_nt_idx0 - half)
    hi = min(len(mrna_seq), middle_nt_idx0 + half + 1)
    return mrna_seq[lo:hi], lo, hi, (hi - lo)

swap_rows = []
for gene in ("GJB2", "GJB6"):
    r = REF[gene]
    cds, cs, mrna = r["cds"], r["cds_start"], r["mrna"]
    n_deg = r["n_deg"]
    cai_ref = r["CAI"]

    for ci in range(r["n_sense_codons"]):
        wt = cds[ci*3:ci*3+3]
        aa = CODON_TABLE[wt]
        alts = sorted(c for c in AA2COD[aa] if c != wt)
        if not alts:
            continue   # Met and Trp have no synonymous alternatives

        mid_idx0 = cs + ci * 3 + 1   # middle nucleotide of codon (0-based in mRNA)
        w_wt, lo, hi, wlen = extract_81nt_window(mrna, mid_idx0, PRIMARY_HALF)
        wt_mfe, wt_gens = fold_ensemble_81nt(w_wt)

        for alt_rank, alt in enumerate(alts, 1):
            diffs = [p for p in range(3) if wt[p] != alt[p]]
            n_nt = len(diffs)
            is_snv = (n_nt == 1)

            # HGVS c. and p. nomenclature
            c_start1 = ci * 3 + 1
            if is_snv:
                d0 = diffs[0]
                c_pos = c_start1 + d0
                c_ref, c_alt = wt[d0], alt[d0]
                variant_c = f"c.{c_pos}{c_ref}>{c_alt}"
                g_pos = cds_to_genomic(gene, c_pos - 1)
                g_ref = c_ref.translate(COMP_TBL) if GENE_STRAND[gene] == -1 else c_ref
                g_alt = c_alt.translate(COMP_TBL) if GENE_STRAND[gene] == -1 else c_alt
            else:
                k0, k1 = diffs[0], diffs[-1]
                c_pos, c_ref, c_alt = np.nan, None, None
                variant_c = f"c.{c_start1 + k0}_{c_start1 + k1}delins{alt[k0:k1+1]}"
                g_pos, g_ref, g_alt = np.nan, None, None

            variant_p = f"p.({AA3[aa]}{ci+1}=)"

            # Fold alternative 81-nt window
            alt_mrna = mrna[:cs + ci*3] + alt + mrna[cs + ci*3 + 3:]
            w_alt, _, _, _ = extract_81nt_window(alt_mrna, mid_idx0, PRIMARY_HALF)
            alt_mfe, alt_gens = fold_ensemble_81nt(w_alt)

            # Whole-transcript CAI after single substitution
            cai_alt = cai_ref * ((W_CAI_REF[alt] / W_CAI_REF[wt]) ** (1.0 / n_deg))

            swap_rows.append(dict(
                gene=gene, transcript=r["acc"], codon_index=ci + 1, aa=aa,
                wt_codon=wt, alt_codon=alt, alt_row=alt_rank,
                variant_c=variant_c, variant_p=variant_p,
                n_nt_changes=n_nt, snv_reachable=is_snv,
                c_pos=c_pos, c_ref=c_ref, c_alt=c_alt,
                g_pos=g_pos, g_ref=g_ref, g_alt=g_alt,
                wt_RSCU_gene=RSCU_GENE[gene][wt], alt_RSCU_gene=RSCU_GENE[gene][alt],
                dRSCU_gene=RSCU_GENE[gene][alt] - RSCU_GENE[gene][wt],
                wt_wCAI=W_CAI_REF[wt], alt_wCAI=W_CAI_REF[alt],
                delta_cai_weight=W_CAI_REF[alt] - W_CAI_REF[wt],
                dCAI_transcript=cai_alt - cai_ref,
                wt_wtAI=TAI_WEIGHTS[wt], alt_wtAI=TAI_WEIGHTS[alt],
                delta_tai_weight=TAI_WEIGHTS[alt] - TAI_WEIGHTS[wt],
                WT_Gens_codon_40=wt_gens, ALT_Gens_codon_40=alt_gens,
                dGens_codon_40=alt_gens - wt_gens,
                WT_MFE_codon_40=wt_mfe, ALT_MFE_codon_40=alt_mfe,
                dMFE_codon_40=alt_mfe - wt_mfe
            ))

swaps_df = pd.DataFrame(swap_rows)
print(f"Enumerated {len(swaps_df)} synonymous alternatives across {swaps_df[swaps_df.gene=='GJB2']['codon_index'].nunique()} GJB2 "
      f"and {swaps_df[swaps_df.gene=='GJB6']['codon_index'].nunique()} GJB6 positions "
      f"({int(swaps_df.snv_reachable.sum())} SNVs, {int((~swaps_df.snv_reachable).sum())} MNVs; "
      f"{swaps_df.groupby(['wt_codon', 'alt_codon']).ngroups} distinct codon replacements).")

# Summary of changes relative to the reference codon across all synonymous codon alternatives
summary_rows = []
for col, m_label, fmt in [
    ("dRSCU_gene",       "ΔRSCU",            ".3f"),
    ("delta_cai_weight", "ΔwCAI",            ".3f"),
    ("delta_tai_weight", "ΔwtAI",            ".3f"),
    ("dGens_codon_40",   "ΔGens (kcal/mol)", ".2f"),
]:
    for g_label in ("GJB2", "GJB6", "Combined"):
        sub = swaps_df if g_label == "Combined" else swaps_df[swaps_df.gene == g_label]
        x = sub[col]
        med = f"{x.median():+{fmt}}".replace("-", "−")
        rng_str = f"{x.min():+{fmt}} to {x.max():+{fmt}}".replace("-", "−")
        summary_rows.append({
            "Measure": m_label,
            "Gene": g_label,
            "n": f"{len(x):,}",
            "Median": med,
            "Range": rng_str,
            "Decrease (%)": f"{100.0 * (x < -1e-12).mean():.1f}",
            "Increase (%)": f"{100.0 * (x > 1e-12).mean():.1f}",
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(WORKDIR, "Summary_Synonymous_Alternatives.csv"), index=False)
print("\n=== Summary of Changes Across All Synonymous Codon Alternatives (Section 3.4) ===")
print(summary_df.to_string(index=False))

# Wobble-pairing penalty sensitivity analysis: 1,000 random parameter sets within +/-0.2 (seed = 2026)
PAIRS = sorted(set(zip(swaps_df["wt_codon"], swaps_df["alt_codon"])))
d_ref = swaps_df["delta_tai_weight"].to_numpy()
top5_ref = set(sorted(PAIRS, key=lambda p: (TAI_WEIGHTS[p[1]] - TAI_WEIGHTS[p[0]], p))[:5])

rng_wob = np.random.default_rng(RANDOM_SEED)
PEN_KEYS = [("sIU", 0.0), ("sGU", 0.41), ("sIC", 0.28), ("sIA", 0.9999), ("sUG", 0.68)]
wob_recs = []
for k in range(1000):
    par = {p: float(np.clip(rng_wob.uniform(v - 0.2, v + 0.2), 0.0, 1.0)) for p, v in PEN_KEYS}
    w_k = compute_tai_weights(tGCN, **par)
    d_k = np.array([w_k[y] - w_k[x] for x, y in zip(swaps_df["wt_codon"], swaps_df["alt_codon"])])
    top5_k = set(sorted(PAIRS, key=lambda p: (w_k[p[1]] - w_k[p[0]], p))[:5])
    wob_recs.append(dict(
        set_id=k, **par,
        pct_negative=100.0 * float((d_k < 0).mean()),
        spearman_vs_baseline=float(stats.spearmanr(d_k, d_ref)[0]),
        top5_retained=len(top5_k & top5_ref)
    ))
wob_df = pd.DataFrame(wob_recs)
wob_df.to_csv(os.path.join(WORKDIR, "tai_random_penalties.csv"), index=False)
lo95, hi95 = np.percentile(wob_df["pct_negative"], [2.5, 97.5])
print(f"\n=== Wobble-Pairing Penalty Sensitivity (1,000 Random Sets, ±0.2, seed={RANDOM_SEED}) ===")
print(f"Central 95% negative ΔwtAI range: {lo95:.1f}% to {hi95:.1f}% (baseline: {(d_ref<0).mean()*100:.1f}%)")
print(f"Median Spearman correlation with baseline: {wob_df['spearman_vs_baseline'].median():.3f}")
print(f"Sets retaining >=3 of the 5 largest baseline decreases: {100*(wob_df['top5_retained']>=3).mean():.1f}%")

# Save TableS1_all_alternatives.csv for downstream primate genome alignment & submission
swaps_df.to_csv(os.path.join(WORKDIR, "TableS1_all_alternatives.csv"), index=False)


In [ ]:
# == Cell 9 == Fig. 4: Positional Maps of ΔRSCU, ΔwCAI, ΔwtAI & ΔGens (3.4) =====
# Drawn at 174 mm width.
from matplotlib.colors import TwoSlopeNorm

PANEL_SPECS = [
    ("dRSCU_gene",       r"$\Delta$RSCU",                         3.75),
    ("delta_cai_weight", r"$\Delta w_{\mathrm{CAI}}$",            0.85),
    ("delta_tai_weight", r"$\Delta w_{\mathrm{tAI}}$",            0.70),
    ("dGens_codon_40",   "$\\Delta G_{\\mathrm{ens}}$\n(kcal/mol)", 7.00),
]
MAX_COD = 261
MAX_ALT = 5
XTICKS = [1, 20, 40, 60, 80, 100, 120, 140, 160, 180, 200, 220, 240, 261]

fig = plt.figure(figsize=(FULL_W, FULL_W * 1.05))
outer_gs = fig.add_gridspec(4, 1, hspace=0.30, left=0.135, right=0.895, top=0.985, bottom=0.06)

cmap_div = plt.cm.RdBu_r.copy()
cmap_div.set_bad(color="#d9d9d9")

for p_idx, (col, ylabel_txt, vlim) in enumerate(PANEL_SPECS):
    inner_gs = outer_gs[p_idx].subgridspec(2, 1, hspace=0.12)
    norm_p = TwoSlopeNorm(vmin=-vlim, vcenter=0.0, vmax=vlim)
    im_ref = None
    for g_idx, gene in enumerate(("GJB2", "GJB6")):
        ax = fig.add_subplot(inner_gs[g_idx])
        n_c = REF[gene]["n_sense_codons"]
        mat = np.full((MAX_ALT, MAX_COD), np.nan)
        sub = swaps_df[swaps_df.gene == gene]
        for r in sub.itertuples():
            mat[int(r.alt_row) - 1, int(r.codon_index) - 1] = getattr(r, col)
        im_ref = ax.imshow(mat[:, :n_c], aspect="auto", cmap=cmap_div, norm=norm_p,
                           interpolation="nearest", extent=(0.5, n_c + 0.5, MAX_ALT + 0.5, 0.5))
        if n_c < MAX_COD:
            ax.axvspan(n_c + 0.5, MAX_COD + 0.5, color="white", zorder=3)
            ax.axvline(n_c + 0.5, color="#555555", ls=":", lw=0.6, zorder=4)
        ax.set_xlim(0.5, MAX_COD + 0.5)
        ax.set_yticks([1, 3, 5])
        ax.set_yticklabels(["1", "3", "5"], fontsize=FS)
        ax.tick_params(axis="both", length=2.5, width=0.5)
        for sp in ax.spines.values():
            sp.set_linewidth(0.5)
        ax.set_ylabel(gene, fontsize=FS, style="italic", rotation=0, labelpad=16, va="center")
        ax.set_xticks(XTICKS)
        if p_idx == 3 and g_idx == 1:
            ax.tick_params(axis="x", labelsize=FS)
            ax.set_xlabel("Codon position")
        else:
            ax.tick_params(labelbottom=False)

    pos_top = fig.axes[-2].get_position()
    pos_bot = fig.axes[-1].get_position()
    cax = fig.add_axes([0.905, pos_bot.y0, 0.012, pos_top.y1 - pos_bot.y0])
    cb = fig.colorbar(im_ref, cax=cax, ticks=[-vlim, 0, vlim])
    cb.ax.set_yticklabels([f"−{vlim:g}", "0", f"{vlim:g}"], fontsize=FS)
    cb.ax.tick_params(length=2.5, width=0.5)
    cb.outline.set_linewidth(0.5)
    fig.text(0.030, (pos_top.y1 + pos_bot.y0) / 2, ylabel_txt,
             va="center", ha="center", rotation=90, fontsize=FS)

fig4_out = save_figure(fig, "Fig4")
plt.show()

In [ ]:
# == Cell 10 == Primate Genome Alignments (LASTZ_NET) & Ancestral Codons (2.3, 3.3) ==
REST_URL = "https://rest.ensembl.org"
FTP_URL  = "https://ftp.ensembl.org/pub"
PAIRWISE_METHOD = "LASTZ_NET"
PARSIMONY_TRIO  = ["pan_troglodytes", "gorilla_gorilla", "pongo_abelii"]
REAL_SPECIES    = re.compile(r"^[a-z]+(_[a-z]+)+$")

CLADE_OF = {
    **{g: "great apes" for g in ["pan", "gorilla", "pongo"]},
    **{g: "gibbons" for g in ["nomascus", "hylobates", "hoolock", "symphalangus"]},
    **{g: "Old World monkeys" for g in ["macaca", "papio", "chlorocebus", "cercocebus", "mandrillus",
                                         "theropithecus", "rhinopithecus", "colobus", "piliocolobus"]},
    **{g: "New World monkeys" for g in ["callithrix", "saimiri", "cebus", "sapajus", "aotus"]},
    **{g: "tarsiers" for g in ["carlito", "tarsius"]},
    **{g: "strepsirrhines" for g in ["microcebus", "otolemur", "propithecus", "prolemur", "lemur"]},
}
CLADES = ["great apes", "gibbons", "Old World monkeys", "New World monkeys", "tarsiers", "strepsirrhines", "outgroup"]
LABEL = {
    "pan_troglodytes": "chimpanzee", "pan_paniscus": "bonobo", "gorilla_gorilla": "gorilla",
    "pongo_abelii": "orangutan", "nomascus_leucogenys": "gibbon", "macaca_mulatta": "rhesus macaque",
    "macaca_fascicularis": "crab-eating macaque", "macaca_nemestrina": "pig-tailed macaque",
    "papio_anubis": "olive baboon", "chlorocebus_sabaeus": "green monkey", "cercocebus_atys": "sooty mangabey",
    "mandrillus_leucophaeus": "drill", "rhinopithecus_roxellana": "golden snub-nosed monkey",
    "rhinopithecus_bieti": "black snub-nosed monkey", "callithrix_jacchus": "marmoset",
    "saimiri_boliviensis_boliviensis": "squirrel monkey", "cebus_imitator": "capuchin",
    "aotus_nancymaae": "owl monkey", "carlito_syrichta": "tarsier", "microcebus_murinus": "mouse lemur",
    "otolemur_garnettii": "galago", "propithecus_coquereli": "sifaka", "prolemur_simus": "bamboo lemur",
    "mus_musculus": "mouse"
}
PAIRWISE_SPECIES = sorted(LABEL.keys())

S = requests.Session()
S.headers.update({"Content-Type": "application/json", "Accept": "application/json"})

def ensembl_rest(path, params=None, cache_name=None):
    cpath = os.path.join(PRIMATE_CACHE, cache_name) if cache_name else None
    if cpath and os.path.exists(cpath):
        return json.load(open(cpath))
    for attempt in range(6):
        r = S.get(REST_URL + path, params=params, timeout=180)
        if r.status_code == 429:
            time.sleep(float(r.headers.get("Retry-After", 2)) + 0.5); continue
        if r.status_code >= 500:
            time.sleep(2 ** attempt); continue
        r.raise_for_status()
        data = r.json()
        if cpath:
            json.dump(data, open(cpath, "w"))
        return data
    raise RuntimeError(f"Ensembl REST failed: {path}")

def parse_pairwise_alignment(data):
    blocks = [data] if isinstance(data, dict) else [b for b in data if isinstance(b, dict) and b.get("alignments")]
    plus = {}
    for b in blocks:
        al = b["alignments"]
        hum = [a for a in al if a.get("species") == "homo_sapiens" and str(a.get("seq_region")) == CHROM]
        if not hum: continue
        h = max(hum, key=lambda a: int(a["end"]) - int(a["start"]))
        seq, st, en, strand = h["seq"].upper(), int(h["start"]), int(h["end"]), int(h.get("strand", 1))
        pos, cols = (st if strand == 1 else en), []
        for ch in seq:
            if ch == "-": cols.append(None)
            else:
                cols.append(pos)
                pos += 1 if strand == 1 else -1
        by_sp = defaultdict(list)
        for a in al:
            sp = a.get("species")
            if sp and sp != "homo_sapiens" and REAL_SPECIES.match(sp):
                by_sp[sp].append(a)
        for sp, entries in by_sp.items():
            best = max(entries, key=lambda a: sum(x == y for x, y in zip(a["seq"].upper(), seq) if x not in "-N"))
            d = plus.setdefault(sp, {})
            for p, ch in zip(cols, best["seq"].upper()):
                if p is not None:
                    d[p] = ch.translate(COMP_TBL) if strand == -1 else ch
    return plus

GENE_SPAN = {
    "GJB2": {"K": 20189582, "codons": 226, "start": 20188901, "end": 20189581},
    "GJB6": {"K": 20223481, "codons": 261, "start": 20222695, "end": 20223480},
}

SPC = {g: {} for g in GENE_SPAN}
for gene, ginfo in GENE_SPAN.items():
    K, n = ginfo["K"], ginfo["codons"]
    region = f"/alignment/region/homo_sapiens/{CHROM}:{ginfo['start']}-{ginfo['end']}:1"
    for sp in PAIRWISE_SPECIES:
        data = ensembl_rest(region, params={"method": PAIRWISE_METHOD, "species_set": ["homo_sapiens", sp], "aligned": 1},
                            cache_name=f"pw_{gene}_{sp}.json")
        plus = parse_pairwise_alignment(data)
        if sp in plus:
            cods = []
            for i in range(1, n + 1):
                c = "".join(plus[sp].get(K - k, "-").translate(COMP_TBL) for k in (3*i - 2, 3*i - 1, 3*i))
                cods.append(c if re.fullmatch(r"[ACGT]{3}", c) else None)
            SPC[gene][sp] = cods

# Tarsier GC3 check
for gene in ("GJB2", "GJB6"):
    t_cods = [c for c in SPC[gene]["carlito_syrichta"] if c is not None]
    t_gc3 = 100.0 * np.mean([c[2] in "GC" for c in t_cods])
    print(f"Tarsier {gene} GC3 = {t_gc3:.1f}% ({len(t_cods)} aligned codons)")

# Ensembl human ancestral sequence (chromosome 13) + parsimony fallback
ANC_FILE = os.path.join(PRIMATE_CACHE, f"homo_sapiens_ancestor_{CHROM}.fa")
if not os.path.exists(ANC_FILE):
    url = f"{FTP_URL}/current_fasta/ancestral_alleles/homo_sapiens_ancestor_GRCh38.tar.gz"
    with requests.get(url, stream=True, timeout=900) as r:
        r.raise_for_status()
        with tarfile.open(fileobj=r.raw, mode="r|gz") as tf:
            for m in tf:
                if re.search(rf"_ancestor_{CHROM}\.fa(\.gz)?$", m.name):
                    raw = tf.extractfile(m).read()
                    if m.name.endswith(".gz"): raw = gzip.decompress(raw)
                    open(ANC_FILE, "wb").write(raw); break

with open(ANC_FILE) as fh:
    hdr = fh.readline().strip()
    anc_seq = "".join(l.strip() for l in fh)

anc_rows = []
for gene, ginfo in GENE_SPAN.items():
    K, n = ginfo["K"], ginfo["codons"]
    for i in range(1, n + 1):
        b = "".join(anc_seq[K - k - 1].translate(COMP_TBL) for k in (3*i - 2, 3*i - 1, 3*i))
        hc = b if re.fullmatch(r"[ACGT]{3}", b) else None
        trio = [SPC[gene][sp][i - 1] for sp in PARSIMONY_TRIO if sp in SPC[gene]]
        pars = trio[0] if (len(trio) == 3 and all(trio) and len(set(trio)) == 1) else None
        anc_cod = hc if hc is not None else pars
        src_lbl = "Ensembl EPO ancestor" if hc is not None else ("parsimony (chimpanzee, gorilla, orangutan)" if pars is not None else "")
        hum_cod = REF[gene]["codons"][i - 1]
        anc_rows.append(dict(
            gene=gene, codon_index=i, human_codon=hum_cod,
            ancestral_codon_hc=hc, parsimony_codon=pars,
            ancestral_codon=anc_cod, ancestral_source=src_lbl,
            human_derived=bool(anc_cod is not None and anc_cod != hum_cod)
        ))
ANC = pd.DataFrame(anc_rows)
ANC.to_csv(os.path.join(PRIMATE_DIR, "ancestral_codons.csv"), index=False)

for gene, d in ANC.groupby("gene"):
    n_tot = int(d["ancestral_codon"].notna().sum())
    n_epo = int(d["ancestral_codon_hc"].notna().sum())
    n_par = int((d["ancestral_codon_hc"].isna() & d["parsimony_codon"].notna()).sum())
    derived = [f"{r.codon_index} {r.ancestral_codon}>{r.human_codon} [{r.ancestral_source.split(' (')[0]}]"
               for r in d[d.human_derived].itertuples()]
    print(f"{gene}: ancestral codons reconstructed = {n_tot}/{len(d)} ({n_epo} EPO + {n_par} parsimony) | "
          f"Human-derived: {', '.join(derived)}")

# Species summary table (SUMM) and classification of all 1,130 synonymous alternatives (EVO)
summ_rows = []
for gene in ("GJB2", "GJB6"):
    h_cods = REF[gene]["codons"]
    for sp, s_cods in SPC[gene].items():
        cl = "outgroup" if sp == "mus_musculus" else CLADE_OF[sp.split("_")[0]]
        n_al = sum(c is not None for c in s_cods)
        same = sum(c == h for h, c in zip(h_cods, s_cods) if c is not None)
        syn  = sum((c != h) and (CODON_TABLE[c] == CODON_TABLE[h]) for h, c in zip(h_cods, s_cods) if c is not None)
        aa_d = n_al - same - syn
        summ_rows.append(dict(
            gene=gene, species=sp, label=LABEL[sp], clade=cl, primate=(cl != "outgroup"),
            codons_aligned=n_al, aa_identity_pct=round(100 * (same + syn) / n_al, 1),
            codon_identity_pct=round(100 * same / n_al, 1),
            synonymous_differences=syn, amino_acid_differences=aa_d,
            no_aligned_codon=len(h_cods) - n_al
        ))
SUMM = pd.DataFrame(summ_rows)
SUMM["clade_rank"] = SUMM["clade"].map({c: i for i, c in enumerate(CLADES)})
SUMM = SUMM.sort_values(["gene", "clade_rank", "codon_identity_pct"], ascending=[True, True, False])
SUMM.drop(columns="clade_rank").to_csv(os.path.join(PRIMATE_DIR, "species_summary_genome.csv"), index=False)

PRIMATES_23 = [sp for sp in PAIRWISE_SPECIES if sp != "mus_musculus"]
anc_lookup = ANC.set_index(["gene", "codon_index"])
evo_rows = []
for r in swaps_df.itertuples():
    g, i = r.gene, int(r.codon_index)
    have = [sp for sp in PRIMATES_23 if SPC[g][sp][i - 1] == r.alt_codon]
    a_row = anc_lookup.loc[(g, i)]
    restores = (a_row.ancestral_codon == r.alt_codon)
    cls = "restores ancestral codon" if restores else ("present in another primate" if have else "not observed in primates")
    evo_rows.append(dict(
        gene=g, variant_c=r.variant_c, variant_p=r.variant_p, codon_index=i,
        wt_codon=r.wt_codon, alt_codon=r.alt_codon, snv=bool(r.snv_reachable),
        ancestral_codon=a_row.ancestral_codon, ancestral_source=a_row.ancestral_source,
        reference_codon_derived=bool(a_row.human_derived),
        primates_with_alt_codon=len(have),
        species_with_alt_codon=", ".join(LABEL[s] for s in have),
        evolutionary_class=cls
    ))
EVO = pd.DataFrame(evo_rows)
EVO.to_csv(os.path.join(PRIMATE_DIR, "alternatives_evolutionary_class.csv"), index=False)
print("\n=== Evolutionary Classification of the 1,130 Synonymous Alternatives ===")
print(pd.crosstab(EVO.evolutionary_class, EVO.snv.map({True: "SNV", False: "MNV"}), margins=True).to_string())
print("\n3 Alternatives Restoring the Inferred Ancestral Codon:")
print(EVO[EVO.evolutionary_class == "restores ancestral codon"][
    ["gene", "variant_c", "variant_p", "wt_codon", "alt_codon", "ancestral_source", "primates_with_alt_codon"]
].to_string(index=False))


In [ ]:
# == Cell 11 == Fig. 3: Amino Acid & Codon Identity Across 23 Primates (3.3) ====
# Panels a (GJB2) and b (GJB6) stacked vertically at 174 mm width.
fig, axes = plt.subplots(2, 1, figsize=(FULL_W, FULL_W * 1.18))
fig.subplots_adjust(hspace=1.05, top=0.93, bottom=0.165, left=0.08, right=0.99)

for k, gene in enumerate(("GJB2", "GJB6")):
    ax = axes[k]
    g = SUMM[SUMM.gene == gene].reset_index(drop=True)
    x = np.arange(len(g))
    for cl, sub in g.groupby("clade", sort=False):
        if CLADES.index(cl) % 2 == 0:
            ax.axvspan(sub.index.min() - 0.5, sub.index.max() + 0.5, color="#ececec", lw=0, zorder=0)
    ax.plot(x, g.aa_identity_pct, "-o", color="black", ms=3.5, lw=1.0,
            label="Amino acid identity", zorder=3)
    ax.plot(x, g.codon_identity_pct, "--s", color="#6e6e6e", mfc="white", mec="#6e6e6e", ms=3.5,
            mew=0.8, lw=1.0, label="Codon identity", zorder=3)
    tr = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
    for xi, (s_, a_) in enumerate(zip(g.synonymous_differences, g.amino_acid_differences)):
        ax.text(xi, 1.03, f"{s_}/{a_}", transform=tr, rotation=90, ha="center", va="bottom", fontsize=FS)
    ax.set_xticks(x)
    ax.set_xticklabels(g.label, rotation=50, ha="right", rotation_mode="anchor", fontsize=FS)
    ax.set_xlim(-0.6, len(g) - 0.4)
    ax.set_ylim(45, 102)
    ax.set_yticks([50, 60, 70, 80, 90, 100])
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_ylabel("Identity with human (%)")
    panel_label(ax, "ab"[k], x=-0.07, y=1.17)
    ax.text(-0.045, 1.17, gene, transform=ax.transAxes, fontsize=FS, style="italic", va="bottom")
    if k == 0:
        ax.legend(loc="lower left", frameon=False, handlelength=2.2)

fig3_out = save_figure(fig, "Fig3")
plt.show()

In [ ]:
# == Cell 12 == ClinVar Synonymous Variants on the Synonymous & Primate Maps (2.5, 3.5) ==
CLINVAR_CSV = os.path.join(WORKDIR, "clinvar_candidates.csv")

if os.path.exists(CLINVAR_CSV):
    cv = pd.read_csv(CLINVAR_CSV)
    # Verify reference bases against CDS
    cv["_cds_base"] = [REF[g]["cds"][int(c) - 1] for g, c in zip(cv["gene"], cv["c_pos"])]
    assert (cv["_cds_base"] == cv["ref"]).all(), "ClinVar reference base mismatch!"
    cv_j = cv.rename(columns={"ref": "c_ref", "alt": "c_alt"})[
        ["gene", "c_pos", "c_ref", "c_alt", "hgvs_c", "clinvar_id", "clinsig", "review"]
    ]
    atlas = swaps_df.merge(cv_j, on=["gene", "c_pos", "c_ref", "c_alt"], how="left")
    atlas["in_clinvar"] = atlas["clinvar_id"].notna()
else:
    raise FileNotFoundError(f"Place {CLINVAR_CSV} in WORKDIR to annotate ClinVar variants.")

atlas = atlas.merge(
    EVO[["gene", "variant_c", "ancestral_codon", "ancestral_source", "reference_codon_derived",
         "primates_with_alt_codon", "species_with_alt_codon", "evolutionary_class"]],
    on=["gene", "variant_c"], how="left"
)
atlas.to_csv(os.path.join(WORKDIR, "TableS1_all_alternatives.csv"), index=False)

cv_mapped = atlas[atlas["in_clinvar"]].copy()
cv_mapped.to_csv(os.path.join(WORKDIR, "ClinVar_Synonymous_Atlas_Mapped.csv"), index=False)

n_cv_tot = len(cv_mapped)
n_cv_g2  = int((cv_mapped.gene == "GJB2").sum())
n_cv_g6  = int((cv_mapped.gene == "GJB6").sum())
n_cv_prim = int((cv_mapped.primates_with_alt_codon > 0).sum())
n_cv_abs  = n_cv_tot - n_cv_prim

print("=== 3.5 Synonymous Variants Reported in ClinVar ===")
print(f"ClinVar synonymous SNVs mapped: {n_cv_tot} total ({n_cv_g2} in GJB2, {n_cv_g6} in GJB6)")
print(f"Observed as orthologous codon in >=1 non-human primate: {n_cv_prim}/{n_cv_tot} ({100*n_cv_prim/n_cv_tot:.1f}%)")
print(f"Absent across all 23 examined primates: {n_cv_abs}/{n_cv_tot} ({100*n_cv_abs/n_cv_tot:.1f}%)")

# Highlighted ClinVar variants in Section 3.5
HIGHLIGHT_CV = [
    ("GJB2", "c.186C>T"),
    ("GJB6", "c.489G>A"),
    ("GJB2", "c.171G>A"),
    ("GJB2", "c.174A>G"),
    ("GJB6", "c.393G>A"),
    ("GJB6", "c.15G>T"),
    ("GJB6", "c.15G>A"),
]
hl_df = atlas[atlas.apply(lambda r: (r["gene"], r["variant_c"]) in HIGHLIGHT_CV, axis=1)][
    ["gene", "variant_c", "variant_p", "wt_codon", "alt_codon",
     "dRSCU_gene", "delta_cai_weight", "delta_tai_weight", "dGens_codon_40",
     "primates_with_alt_codon", "evolutionary_class"]
]
print("\nHighlighted Manuscript Variants (Section 3.5):")
print(hl_df.round({"dRSCU_gene": 2, "delta_cai_weight": 3, "delta_tai_weight": 3, "dGens_codon_40": 2}).to_string(index=False))

# Record the software environment used for this run
with open(os.path.join(WORKDIR, "run_metadata_manuscript.json"), "w") as fh:
    json.dump(RUN_METADATA, fh, indent=2)

In [ ]:
# == Cell 13 == Electronic Supplementary Material: Online Resources 1 and 2 ========
# Springer requires supplementary files to be named consecutively (ESM_1, ESM_2) and
# each file to carry the article title, journal, author names and the affiliation and
# e-mail address of the corresponding author.
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment

ARTICLE_TITLE = ("Codon Architecture of the Paralogous Connexin Genes GJB2 and GJB6 and a "
                 "Complete Map of Their Synonymous Alternatives at the DFNB1 Locus")
JOURNAL = "Molecular Genetics and Genomics"
AUTHORS = "Seda Nur Homurlu, Oğuz Altungöz"
CORRESPONDING = ("Corresponding author: Oğuz Altungöz, Dokuz Eylül University, Faculty of Medicine, "
                 "Department of Medical Biology and Genetics, 35340 Izmir, Türkiye; "
                 "oguz.altungoz@deu.edu.tr")

COLUMN_DESCRIPTIONS = {
    "gene": "Gene symbol",
    "transcript": "RefSeq transcript",
    "codon_index": "Codon position in the coding sequence (1 = initiator Met)",
    "aa": "Encoded amino acid (one-letter code)",
    "wt_codon": "Reference codon",
    "alt_codon": "Synonymous alternative codon",
    "alt_row": "Index of the alternative at this position (row in Fig. 4)",
    "variant_c": "HGVS coding DNA description",
    "variant_p": "HGVS protein description",
    "n_nt_changes": "Number of nucleotides changed",
    "snv_reachable": "TRUE = single-nucleotide variant (SNV); FALSE = multinucleotide variant (MNV)",
    "c_pos": "Coding DNA position of the changed nucleotide (SNVs only)",
    "c_ref": "Reference nucleotide, coding strand (SNVs only)",
    "c_alt": "Alternative nucleotide, coding strand (SNVs only)",
    "g_pos": "GRCh38 chromosome 13 position (SNVs only)",
    "g_ref": "Reference nucleotide, genomic plus strand (SNVs only)",
    "g_alt": "Alternative nucleotide, genomic plus strand (SNVs only)",
    "wt_RSCU_gene": "Gene-specific RSCU of the reference codon",
    "alt_RSCU_gene": "Gene-specific RSCU of the alternative codon",
    "dRSCU_gene": "ΔRSCU (alternative minus reference)",
    "wt_wCAI": "Genome-wide codon adaptation weight (wCAI) of the reference codon",
    "alt_wCAI": "wCAI of the alternative codon",
    "delta_cai_weight": "ΔwCAI (alternative minus reference)",
    "dCAI_transcript": "Change in gene CAI caused by the single substitution",
    "wt_wtAI": "tRNA adaptation weight (wtAI) of the reference codon",
    "alt_wtAI": "wtAI of the alternative codon",
    "delta_tai_weight": "ΔwtAI (alternative minus reference)",
    "WT_Gens_codon_40": "Ensemble free energy of the 81-nt reference window (kcal/mol)",
    "ALT_Gens_codon_40": "Ensemble free energy of the 81-nt alternative window (kcal/mol)",
    "dGens_codon_40": "ΔGens (alternative minus reference, kcal/mol; positive = destabilization)",
    "WT_MFE_codon_40": "Minimum free energy of the 81-nt reference window (kcal/mol)",
    "ALT_MFE_codon_40": "Minimum free energy of the 81-nt alternative window (kcal/mol)",
    "dMFE_codon_40": "ΔMFE (alternative minus reference, kcal/mol)",
    "hgvs_c": "ClinVar variant name",
    "clinvar_id": "ClinVar VariationID",
    "clinsig": "ClinVar germline classification",
    "review": "ClinVar review status",
    "in_clinvar": "TRUE = variant reported in ClinVar",
    "ancestral_codon": "Inferred human–chimpanzee ancestral codon",
    "ancestral_source": "Source of the ancestral codon (Ensembl EPO or parsimony)",
    "reference_codon_derived": "TRUE = human reference codon differs from the ancestral codon",
    "primates_with_alt_codon": "Number of the 23 non-human primates carrying the alternative codon",
    "species_with_alt_codon": "Primates carrying the alternative codon",
    "evolutionary_class": "Evolutionary classification of the alternative",
    "wt_codon_rarely_used": "TRUE = reference codon is rarely used (wCAI < 0.5)",
    "alt_codon_rarely_used": "TRUE = alternative codon is rarely used (wCAI < 0.5)",
}

def write_esm(df, path, resource_no, description):
    missing = [c for c in df.columns if c not in COLUMN_DESCRIPTIONS]
    assert not missing, f"undocumented columns: {missing}"
    header = [f"Online Resource {resource_no}", ARTICLE_TITLE, JOURNAL, AUTHORS, CORRESPONDING, description]
    start = len(header) + 1
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        df.to_excel(xw, sheet_name=f"Online Resource {resource_no}", index=False, startrow=start)
        pd.DataFrame({"Column": list(df.columns),
                      "Description": [COLUMN_DESCRIPTIONS[c] for c in df.columns]}
                     ).to_excel(xw, sheet_name="Column descriptions", index=False)
    wb = load_workbook(path)
    ws = wb[f"Online Resource {resource_no}"]
    for i, line in enumerate(header, 1):
        ws.cell(row=i, column=1, value=line).font = Font(bold=(i == 1))
    for cell in ws[start + 1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(wrap_text=True, vertical="top")
    ws.freeze_panes = ws.cell(row=start + 2, column=1)
    for col in ws.iter_cols(min_row=start + 1, max_row=start + 1):
        ws.column_dimensions[col[0].column_letter].width = 16
    ws2 = wb["Column descriptions"]
    for cell in ws2[1]:
        cell.font = Font(bold=True)
    ws2.column_dimensions["A"].width = 26
    ws2.column_dimensions["B"].width = 90
    wb.save(path)
    print(f"Saved {os.path.basename(path)}: {len(df)} rows x {df.shape[1]} columns")

# Online Resource 1 (ESM_1.xlsx): all 1,130 synonymous alternatives
esm1 = atlas.copy()
esm1["wt_codon_rarely_used"]  = esm1["wt_wCAI"] < 0.5
esm1["alt_codon_rarely_used"] = esm1["alt_wCAI"] < 0.5
esm1 = esm1.sort_values(["gene", "codon_index", "alt_row"]).reset_index(drop=True)
assert len(esm1) == 1130
write_esm(esm1, os.path.join(WORKDIR, "ESM_1.xlsx"), 1,
          "Complete enumeration of all 1130 single-codon synonymous alternatives in GJB2 and GJB6, "
          "with their ΔRSCU, ΔwCAI, ΔwtAI and ΔGens values, evolutionary classification and ClinVar annotation.")

# Online Resource 2 (ESM_2.xlsx): ClinVar synonymous variants with uncertain or conflicting classifications
ESM2_COLS = ["gene", "transcript", "codon_index", "aa", "variant_c", "variant_p", "clinvar_id", "clinsig",
             "review", "wt_codon", "alt_codon", "wt_codon_rarely_used", "alt_codon_rarely_used",
             "wt_RSCU_gene", "alt_RSCU_gene", "dRSCU_gene", "wt_wCAI", "alt_wCAI", "delta_cai_weight",
             "wt_wtAI", "alt_wtAI", "delta_tai_weight", "dGens_codon_40",
             "primates_with_alt_codon", "species_with_alt_codon", "evolutionary_class"]
unresolved = esm1["in_clinvar"].fillna(False).astype(bool) & \
             esm1["clinsig"].str.contains("Uncertain|Conflicting", case=False, na=False)
esm2 = esm1.loc[unresolved, ESM2_COLS].reset_index(drop=True)
n_vus = int(esm2.clinsig.str.contains("Uncertain").sum())
n_conf = int(esm2.clinsig.str.contains("Conflicting").sum())
print(f"Unresolved ClinVar variants: {len(esm2)} ({n_vus} uncertain significance, {n_conf} conflicting); "
      f"rarely used reference codon: {int(esm2.wt_codon_rarely_used.sum())}; "
      f"rarely used alternative codon: {int(esm2.alt_codon_rarely_used.sum())}")
write_esm(esm2, os.path.join(WORKDIR, "ESM_2.xlsx"), 2,
          "Synonymous variants reported in ClinVar with uncertain or conflicting classifications, "
          "with their codon usage, tRNA adaptation and local folding energy changes.")